# Supervised reproduction: Figure 4 tree regularization

This notebook reproduces the tree-regularization example from the supervised section of the main article:

> Vallerio, M., del Rio Chanona, A., & Navarro-Brull, F. J. (2026). *All you need is noise - from feature selection to explainable industrial AI*. **Digital Chemical Engineering, 18**, 100290.
> Local article copy: `/Users/b42549592/Documents/databelts/COIQCV/publications/2026-Noise.pdf`.

The point of Figure 4 is to show how a synthetic-noise feature can act as an explicit stopping rule for an interpretable tree. The notebook first restricts the model to the selected variables used for the figure: `Delta[PressureC1]`, `FlowC1`, `Temp1`, and `Random Uniform Noise`. It then grows one-estimator LightGBM trees with increasing split caps.

The article's logic is reproduced as follows: Figure 4a shows training R2 continuing to improve up to 40 splits; Figure 4b shows the 40-split tree; Figure 4c shows the smaller tree at the first noise split, 12 splits; and Figure 4d compares feature importance for the 12-split and 40-split trees.

In [ ]:
# Self-contained runtime setup.
# The notebook installs only the packages it needs when they are missing.
# scikit-learn is intentionally not used; LightGBM can handle missing predictor
# values directly, so rows with NaN predictors are not discarded.
import importlib.util
import subprocess
import sys
import textwrap


def ensure(import_name, package_name=None):
    """Install a Python package only when its import is unavailable."""
    package_name = package_name or import_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


for import_name, package_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("openpyxl", "openpyxl"),
    ("matplotlib", "matplotlib"),
    ("lightgbm", "lightgbm"),
]:
    ensure(import_name, package_name)

try:
    import lightgbm  # noqa: F401
except OSError as exc:
    # On macOS, pip-installed LightGBM may need OpenMP at runtime.
    # Raising an explicit message is clearer than the raw dynamic-library error.
    if "libomp" in str(exc):
        raise RuntimeError(textwrap.dedent('''
        LightGBM is installed, but macOS cannot find `libomp.dylib`.
        Install OpenMP once and rerun the notebook:

            mamba install -c conda-forge llvm-openmp lightgbm

        or:

            brew install libomp
        ''')) from exc
    raise

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

warnings.filterwarnings("ignore", category=UserWarning)

# Use one plotting style for every reproduced figure.
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})


# Locate the workbook whether the notebook is run from this folder or elsewhere.
def find_project_root():
    candidates = [
        Path.cwd(),
        Path("/Users/b42549592/Documents/GitHub/all-you-need-is-noise/01_supervised"),
    ]
    for candidate in candidates:
        if (candidate / "distillation_tower_noise_paper.xlsx").exists():
            return candidate
    raise FileNotFoundError("Could not locate distillation_tower_noise_paper.xlsx")


ROOT = find_project_root()
DATA_PATH = ROOT / "distillation_tower_noise_paper.xlsx"
OUTPUT_DIR = ROOT / "results" / "python" / "trees" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the starting data exactly once. LightGBM receives NaNs unchanged later.
raw = pd.read_excel(DATA_PATH, sheet_name="distillation_tower_noise_paper")
raw = raw.sort_values("Date").reset_index(drop=True)

# Process measurements used as candidate supervised predictors.
sensor_features = [
    "OC1", "Temp11", "Temp12", "PressureC1", "TempC1", "Temp1", "FlowC1",
    "Temp2", "Temp3", "TempC2", "TempC3", "Temp4", "Temp5", "Temp6",
    "Temp7", "Temp8", "FlowC9", "FlowC2", "Temp9", "Temp10", "FlowC3",
    "FlowC4", "VapourPressure",
]
# Synthetic-noise features provide a built-in noise floor for feature importance.
snf_features = ["Shuffle[yield]", "Random Uniform Noise", "Random Normal Noise"]
feature_cols = sensor_features + snf_features

print(f"LightGBM {lgb.__version__}")
print(f"Rows: {len(raw)}")
print(f"Data: {DATA_PATH}")

## Figure 4: stopping at the noise threshold

The selected variables represent a compact tree-building example: three process-derived predictors and one synthetic-noise predictor. The scan below fits one tree for each maximum split count from 1 to 40. A larger tree can keep improving training R2, but once the tree starts splitting on the random-noise variable, the article treats that as evidence that further growth is entering noise-fitting territory.

For this reproduction, the first noise split appears at 12 splits, while the unrestricted comparison tree is shown at 40 splits.

In [ ]:
# Variables selected for Figure 4: three signal candidates plus one known-noise feature.
selected_features = ["Delta[PressureC1]", "FlowC1", "Temp1", "Random Uniform Noise"]
noise_feature = "Random Uniform Noise"
target_col = "yield"


# Training R2 is computed directly with NumPy to keep the notebook independent of scikit-learn.
def r2_score_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    return float(1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2))


# Only rows with an observed target are required; predictor NaNs remain available to LightGBM.
def make_xy(features=selected_features):
    y_series = raw[target_col].astype(float)
    keep = y_series.notna()
    X = raw.loc[keep, features].astype(float)
    y = y_series.loc[keep].to_numpy(dtype=float)
    return X, y


# Extract split counts and gain portions for the selected variables.
def importance_for(model, feature_names):
    split = model.feature_importance("split").astype(int)
    gain = model.feature_importance("gain").astype(float)
    total_gain = gain.sum()
    return pd.DataFrame({
        "Term": feature_names,
        "split_count": split,
        "gain_portion": np.where(total_gain > 0, gain / total_gain, 0),
        "is_noise": [name == noise_feature for name in feature_names],
    }).sort_values("gain_portion", ascending=False).reset_index(drop=True)


# Fit a single LightGBM tree whose maximum number of splits is controlled by num_leaves.
# min_data_in_leaf=3 keeps the tree able to reach the 12-split noise threshold on this dataset.
def train_tree(split_cap, features=selected_features, seed=2026, min_data_in_leaf=3):
    X, y = make_xy(features)
    dataset = lgb.Dataset(
        X.to_numpy(), label=y,
        # Safe feature names avoid LightGBM parsing issues with brackets in Delta[PressureC1].
        feature_name=[f"f{i:02d}" for i in range(X.shape[1])],
        free_raw_data=False,
    )
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "seed": seed,
        "feature_fraction_seed": seed,
        "bagging_seed": seed,
        "data_random_seed": seed,
        "deterministic": True,
        "force_col_wise": True,
        "boosting_type": "gbdt",
        "learning_rate": 1.0,
        "feature_fraction": 1.0,
        "num_leaves": split_cap + 1,
        "min_data_in_leaf": min_data_in_leaf,
        "min_gain_to_split": 0.0,
    }
    model = lgb.train(params, dataset, num_boost_round=1)
    pred = model.predict(X.to_numpy())
    importance = importance_for(model, features)
    noise_splits = int(importance.loc[importance["Term"] == noise_feature, "split_count"].iloc[0])
    return {
        "split_cap": split_cap,
        "feature_names": features,
        "actual_splits": int(importance["split_count"].sum()),
        "training_R2": r2_score_np(y, pred),
        "noise_splits": noise_splits,
        "model": model,
        "prediction": pred,
        "y": y,
        "importance": importance,
    }


# Scan split caps from 1 to 40 so panel 4a can show the full growth path.
scan = [train_tree(split_cap) for split_cap in range(1, 41)]
scan_table = pd.DataFrame([
    {k: v for k, v in row.items() if k not in ["model", "prediction", "y", "importance", "feature_names"]}
    for row in scan
])
# The first tree that uses the random-noise feature defines the stopping point.
first_noise_cap = int(scan_table.loc[scan_table["noise_splits"] > 0, "split_cap"].iloc[0])
noise_tree_cap = first_noise_cap
full_cap = 40
noise_tree = scan[noise_tree_cap - 1]
full_tree = scan[full_cap - 1]

print(f"Selected variables: {selected_features}")
print(f"First noise split appears at split cap {first_noise_cap}.")
print(f"Figure 4b full tree: {full_tree['actual_splits']} splits.")
print(f"Figure 4c noise-threshold tree: {noise_tree['actual_splits']} splits, noise splits = {noise_tree['noise_splits']}.")
display(scan_table.head(15))


In [ ]:
# Recursively convert a LightGBM tree dictionary into node/edge records for plotting.
def collect_tree_nodes(node, feature_names, depth=0, nodes=None, edges=None, leaf_counter=None, max_depth=None):
    if nodes is None:
        nodes, edges, leaf_counter = [], [], [0]
    node_id = len(nodes)
    nodes.append({"id": node_id, "x": 0, "y": -depth, "label": "", "color": "#FFFFFF"})
    is_leaf = "leaf_value" in node or (max_depth is not None and depth >= max_depth)
    if is_leaf:
        x = leaf_counter[0]
        leaf_counter[0] += 1
        value = node.get("leaf_value", node.get("internal_value", 0.0))
        label = f"leaf\n{value:.2f}" if "leaf_value" in node else "..."
        color = "#EEF3F7"
    else:
        feature = feature_names[node["split_feature"]]
        threshold = node["threshold"]
        label = f"{feature}\n<= {threshold:.3g}"
        color = "#F7E1DE" if feature == noise_feature else "#E7F0F8"
        left_id = collect_tree_nodes(node["left_child"], feature_names, depth + 1, nodes, edges, leaf_counter, max_depth)
        right_id = collect_tree_nodes(node["right_child"], feature_names, depth + 1, nodes, edges, leaf_counter, max_depth)
        x = (nodes[left_id]["x"] + nodes[right_id]["x"]) / 2
        edges.extend([(node_id, left_id, "yes"), (node_id, right_id, "no")])
    nodes[node_id].update({"x": x, "label": label, "color": color})
    return node_id


# Draw the converted tree using Matplotlib text boxes.
# Noise splits are tinted red; process-variable splits are tinted blue.
def draw_tree(ax, result, title, max_depth=None, fontsize=6.2):
    tree = result["model"].dump_model()["tree_info"][0]["tree_structure"]
    nodes, edges = [], []
    collect_tree_nodes(tree, result["feature_names"], nodes=nodes, edges=edges, leaf_counter=[0], max_depth=max_depth)
    by_id = {node["id"]: node for node in nodes}
    for parent, child, edge_label in edges:
        p, c = by_id[parent], by_id[child]
        ax.plot([p["x"], c["x"]], [p["y"], c["y"]], color="#5E6670", lw=0.72, zorder=1)
        ax.text((p["x"] + c["x"]) / 2, (p["y"] + c["y"]) / 2, edge_label, fontsize=5.2, color="#555555")
    for node in nodes:
        ax.text(
            node["x"], node["y"], node["label"], ha="center", va="center", fontsize=fontsize,
            bbox=dict(boxstyle="round,pad=0.24", facecolor=node["color"], edgecolor="#66717D", linewidth=0.65),
            zorder=2,
        )
    ax.set_title(title)
    ax.set_axis_off()


# Compose the final four-panel Figure 4 image.
fig = plt.figure(figsize=(18, 13), constrained_layout=True)
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.7], width_ratios=[1.0, 1.0])

ax_r2 = fig.add_subplot(gs[0, 0])
ax_imp = fig.add_subplot(gs[0, 1])
ax_full = fig.add_subplot(gs[1, 0])
ax_noise = fig.add_subplot(gs[1, 1])

# 4a: Training R2 keeps increasing as split capacity grows to 40.
ax_r2.plot(scan_table["split_cap"], scan_table["training_R2"], marker="o", color="#2F6F9F", lw=1.45)
ax_r2.axvline(noise_tree_cap, color="#C83E35", ls="--", lw=1.35, label=f"noise first appears at {noise_tree_cap} splits")
ax_r2.set_title("(a) Training R2 as the tree grows to 40 splits")
ax_r2.set_xlabel("Maximum splits")
ax_r2.set_ylabel("Training R2")
ax_r2.set_xlim(0, 41)
ax_r2.legend(frameon=False)

# 4d: Compare the stopped tree against the 40-split tree, ordered by the stopped tree.
importance_12 = noise_tree["importance"].set_index("Term")
importance_40 = full_tree["importance"].set_index("Term")
ordered_terms = importance_12.sort_values("gain_portion", ascending=True).index.tolist()
y = np.arange(len(ordered_terms))
ax_imp.barh(y - 0.18, importance_12.loc[ordered_terms, "gain_portion"], height=0.34, color="#2F6F9F", label="12-split tree")
ax_imp.barh(y + 0.18, importance_40.loc[ordered_terms, "gain_portion"], height=0.34, color="#8D6BB8", label="40-split tree")
for term, ypos in zip(ordered_terms, y):
    if term == noise_feature:
        ax_imp.get_yticklabels()
        ax_imp.text(0, ypos, "", color="#C83E35")
ax_imp.set_yticks(y)
ax_imp.set_yticklabels(ordered_terms)
for tick, term in zip(ax_imp.get_yticklabels(), ordered_terms):
    if term == noise_feature:
        tick.set_color("#C83E35")
        tick.set_fontweight("bold")
ax_imp.set_title("(d) Feature importance, ordered by the 12-split tree")
ax_imp.set_xlabel("Gain portion")
ax_imp.legend(frameon=False)

# 4b and 4c: Tree visualizations.
# The 40-split tree is depth-limited so the panel stays readable;
# the 12-split noise-threshold tree is shown completely.
draw_tree(ax_full, full_tree, "(b) Full tree, 40 splits (depth-limited view)", max_depth=5, fontsize=5.4)
draw_tree(ax_noise, noise_tree, "(c) Tree stopping at noise, 12 splits", max_depth=None, fontsize=5.9)

fig.suptitle("Figure 4. Noise-threshold tree growth and feature importance", fontsize=15)
fig.savefig(OUTPUT_DIR / "figure_4_selected_variable_trees.png", bbox_inches="tight")
plt.show()

scan_table.to_csv(OUTPUT_DIR / "figure_4_selected_variable_split_scan.csv", index=False)
full_tree["importance"].to_csv(OUTPUT_DIR / "figure_4_full_tree_importance.csv", index=False)
noise_tree["importance"].to_csv(OUTPUT_DIR / "figure_4_12_split_tree_importance.csv", index=False)


**Figure 4 caption.** Tree regularization using a synthetic-noise stopping rule. (a) Training R2 improves as the split cap increases through 40 splits. (b) The larger 40-split tree shows the high-capacity fit. (c) The smaller tree stops at the first use of `Random Uniform Noise`, which occurs at 12 splits in this reproduction. (d) Feature importance compares the 12-split and 40-split trees, ordered from most to least important according to the 12-split tree.